In [1]:
# ------- IMPORT LIBRARIES -------

import json
from bs4 import BeautifulSoup, Tag
import requests
import re
from display_helpers import pretty_print_page
from concurrent.futures import ThreadPoolExecutor
from data_helpers import load_cache, save_cache, merge_article_into_cache
import time
from datetime import datetime, timezone
import os
from urllib.parse import urlparse, unquote
from typing import Union, List, Optional
import sqlite3
from random import sample, choices
import pandas as pd



In [2]:
# --------------------- SETUP DATABASE ------------------

# set working directory
os.chdir('/home/schmi/projects/explain2me/data')

# Connect to DB (create if does not exist)
conn = sqlite3.connect(database="WikipediaOne.db")
cur = conn.cursor()

# create tables 'pages' and 'definitions'.
# 'definitions' contains the actual page content.
# 'pages' serves as a lookup table for 'definitions'
cur.execute("""
            CREATE TABLE IF NOT EXISTS pages (
			id INTEGER PRIMARY KEY,
			title TEXT UNIQUE NOT NULL,
			has_simple INTEGER DEFAULT 0,
			has_technical INTEGER DEFAULT 0,
			has_kids INTEGER DEFAULT 0
            );
            """)

cur.execute("""
            CREATE TABLE IF NOT EXISTS definitions (
			id INTEGER PRIMARY KEY,
			page_id INTEGER NOT NULL,
			kind TEXT CHECK(kind IN ('simple', 'technical', 'kids')),
			content TEXT NOT NULL,
			source TEXT,
			created_at TEXT,
			FOREIGN KEY (page_id) REFERENCES pages(id),
			UNIQUE (page_id, kind)
            );
			""")

conn.commit()
conn.close()

In [3]:
# ------------- HELPERS -------------
# (try to) convert simple wiki page url to its classic wiki page url 
# counterpart
# Goal -> have a normal wikipedia page for each simple wikipedia page 
#         already stored

def simple2normalwiki_url(simple_url):
    page_url_segment = simple_url.split("/")[-1]
    normalwiki_base = "https://en.wikipedia.org/wiki/"
    normalwiki_url = normalwiki_base + page_url_segment
    return normalwiki_url


#-----------------------------
# And conversely....

def normal2simplewiki_url(normal_url):
    page_url_segment = normal_url.split("/")[-1]
    simplewiki_base = "https://simple.wikipedia.org/wiki/"
    simplewiki_url = simplewiki_base + page_url_segment
    return simplewiki_url


# Normal Wikipedia page scraper

In [4]:
# ----------------------------------------
# 			  SCRAPING METHOD
# ----------------------------------------

# ---------------- CONFIG ----------------
IGNORE_CLASSES = {
    "sidebar-list", "navbar", "infobox", "toc",
    "thumb", "mw-default-size", "metadata"
}

STOP_SECTIONS = {
    "references", "external links", "see also", "notes", "further reading"
}


# ---------------- HELPERS ----------------
def clean_paragraph(el: Tag) -> str:
    """Clean paragraph text, preserving math as LaTeX."""

    # Remove citation markers
    for sup in el.find_all("sup"):
        sup.decompose()

    # Preserve math
    for math in el.find_all("math"):
        latex = math.get("alttext") or math.get_text(strip=True)
        latex = latex.strip()

        is_block = el.get_text(strip=True) == math.get_text(strip=True)
        math.replace_with(
            f"\n$$\n{latex}\n$$\n" if is_block else f"${latex}$"
        )

    # Lists
    if el.name in {"ul", "ol"}:
        lines = []
        for i, li in enumerate(el.find_all("li", recursive=False), start=1):
            txt = clean_paragraph(li)
            if txt:
                lines.append(f"- {txt}" if el.name == "ul" else f"{i}) {txt}")
        return "\n".join(lines)

    # Text cleanup
    text = el.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return text.strip()


def is_ignored(el: Tag) -> bool:
    """Ignore elements inside navboxes, infoboxes, thumbnails, TOC."""
    for parent in el.parents:
        classes = parent.get("class", [])
        if any(cls in IGNORE_CLASSES for cls in classes):
            return True
    return False



# ---------------- MAIN SCRAPER ----------------
def scrape_normal_wiki(url: str) -> dict:
    headers = {"User-Agent": "ReverseMentorBot/0.1"}
    
    try:
         res = requests.get(url, headers=headers, timeout=10)
         # Raise an HTTPError for 4xx/5xx responses (e.g., 404, 500, 429), ensuring failed HTTP responses are treated as errors.
         res.raise_for_status()
		
    except requests.RequestException as e:
        # catches all request-related failures: connection errors, timeouts, invalid URLs, and HTTP errors raised by raise_for_status()
		# i.e. network/environment-level failures, not parsing or scraper-logic errors.
        raise RuntimeError(f"Failed to fetch URL: {url}") from e


    soup = BeautifulSoup(res.text, "html.parser")


    content = soup.find("div", id="mw-content-text")
    
    if content is None:
        raise ValueError(f"Content div not found for {url}")

    sections = []
    intro = None
    current = None

    # Traverse in DOM order
    for el in content.find_all(
        ["p", "li", "dd", "ul", "ol", "h2", "h3", "h4", "h5"],
        recursive=True
    ):
        if is_ignored(el):
            continue

        # ---------- HEADINGS ----------
        if el.name.startswith("h"):
            heading = el.get_text(" ", strip=True).replace("[edit]", "")
            if heading.lower() in STOP_SECTIONS:
                break

            current = {"heading": heading, "paragraphs": []}
            sections.append(current)
            continue

        # ---------- CONTENT ----------
        text = clean_paragraph(el)
        if not text:
            continue

        if current is None:
            if intro is None:
                intro = {"heading": "Introduction", "paragraphs": []}
                sections.insert(0, intro)
            intro["paragraphs"].append(text)
        else:
            current["paragraphs"].append(text)

    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None
    

	# --------- CATEGORY DATA ----------
    categories = []
    category_urls = []

    catlinks = soup.select("#mw-normal-catlinks ul li a")
    for cat in catlinks:
        categories.append(cat.get_text(strip=True))
        href = cat.get("href")
        if href and href.startswith("/wiki/"):
            category_urls.append("https://en.wikipedia.org" + href)



    return {
        "url": url,
        "title": title,
        "sections": sections,
        "categories": categories,
        "category_urls": category_urls,
    }



In [5]:
test_scrape_normal_wiki1 = scrape_normal_wiki(url='https://en.wikipedia.org/wiki/Graph_database')
test_scrape_normal_wiki1

{'url': 'https://en.wikipedia.org/wiki/Graph_database',
 'title': 'Graph database',
 'sections': [{'heading': 'Introduction',
   'paragraphs': ['A graph database ( GDB ) is a database that uses graph structures for semantic queries with nodes, edges, and properties to represent and store data. A key concept of the system is the graph (or edge or relationship). The graph relates the data items in the store to a collection of nodes and edges, the edges representing the relationships between the nodes. The relationships allow data in the store to be linked together directly and, in many cases, retrieved with one operation. Graph databases hold the relationships between data as a priority. Querying relationships is fast because they are perpetually stored in the database. Relationships can be intuitively visualized using graph databases, making them useful for heavily inter-connected data.',
    'Graph databases are commonly referred to as a NoSQL database. Graph databases are similar to 1

# Simple wiki page scraper

In [6]:

# ---------------- HELPERS ----------------

def clean_spaces(text):
    return " ".join(text.split())

def clean_paragraph(el: Tag):
    """
    Clean paragraph text, preserving formulas as LaTeX.
    Handles <p>, <li>, <dd>, <ul>, <ol> elements.
    """

    # Remove citation superscripts
    for sup in el.find_all("sup"):
        sup.decompose()

    # Replace <math> elements with LaTeX
    for math in el.find_all("math"):
        latex = math.get("alttext") or "".join(math.strings).strip()
        math.replace_with(f"${latex.strip()}$")

    # Handle lists
    if el.name in ["ul", "ol"]:
        items = []
        for i, li in enumerate(el.find_all("li", recursive=False), start=1):
            li_text = clean_paragraph(li)
            if li_text:
                items.append(f"- {li_text}" if el.name == "ul" else f"{i}) {li_text}")
        return "\n".join(items)

    # Handle list items and description items
    if el.name in ["li", "dd"]:
        parts = []
        for child in el.children:
            if isinstance(child, Tag):
                parts.append(clean_paragraph(child))
            else:
                parts.append(str(child))
        text = " ".join(filter(None, parts))
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s+([.,;:!?])", r"\1", text)
        return text.strip()

    # Default text
    text = el.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return text.strip()


# --- Recursive content iterator ---
def iter_content_elements(el):
    """
    Yield all relevant content elements in document order.
    Skip navboxes, tables, scripts, styles.
    """
    for child in el.children:
        if not isinstance(child, Tag):
            continue

        if child.name in ["table", "script", "style"]:
            continue

        # Skip sideboxes, navboxes, metadata
        if child.name == "div":
            classes = child.get("class") or []
            if any(c in ["navbox", "vertical-navbox", "metadata", "mbox"] for c in classes):
                continue
            yield from iter_content_elements(child)
            continue
        
        # Skip geo/coordinates spans
        if child.name == "span" and any(c in ["geo", "coordinates"] for c in (child.get("class") or [])):
            continue

		# Recursively yield from spans (other inline containers)
        if child.name == "span":
            yield from iter_content_elements(child)
            continue

        # Yield headings and paragraph-like content
        if child.name in ["p", "ul", "ol", "dd"] + [f"h{i}" for i in range(2, 7)]:
            yield child
        else:
            yield from iter_content_elements(child)





In [7]:

# ---------------- MAIN SCRAPER ----------------

def scrape_simple_wiki(url):
    """
    Scrapes a Simple Wikipedia page and returns structured article data.
    Handles redirects, missing content, and network errors.
    """
    
    headers = {
        "User-Agent": "ReverseMentorBot/0.1 (https://yourdomain.com/contact)"
    }

    stop_sections = {
        "references",
        "other websites",
        "related pages",
        "further reading",
        "external links",
        "see also",
    }

    # --- Fetch page with network error handling ---
    try:
        res = requests.get(url, headers=headers, timeout=10)
        # Raises for 4xx/5xx responses
        res.raise_for_status()
    except requests.RequestException as e:
        # Network/environment-level failure: connection, timeout, HTTP error, invalid URL
        raise RuntimeError(f"Failed to fetch Simple Wikipedia URL: {url}") from e

    soup = BeautifulSoup(res.text, "html.parser")

    # --- Handle redirects ---
    redirect_div = soup.find("div", class_="redirectMsg")
    if redirect_div and redirect_div.find("a"):
        redirect_url = "https://simple.wikipedia.org" + redirect_div.find("a")["href"]
        return scrape_simple_wiki(redirect_url)


    # --- Main content ---
    content = soup.find("div", class_="mw-parser-output")
    if content is None:
        # Page layout changed or empty page
        raise ValueError(f"Main content not found for {url}")

    # --- Article title with fallback ---
    title_tag = soup.find("h1", id="firstHeading")
    if title_tag:
        title = title_tag.get_text(strip=True)
    else:
        # fallback: use last segment of URL
        title = url.split("/")[-1].replace("_", " ")

    article_data = {
        "url": url,
        "title": title,
        "sections": [],
        "categories": [],
        "category_urls": [],
    }

    # --- Introduction section ---
    intro_section = {"heading": "Introduction", "paragraphs": []}
    current_section = intro_section

    # --- Walk content recursively ---
    for el in iter_content_elements(content):
        # Headings start new sections
        if el.name.startswith("h"):
            heading = el.get_text(" ", strip=True).replace("[edit]", "")
            if heading.lower() in stop_sections:
                break
            if intro_section["paragraphs"] and intro_section not in article_data["sections"]:
                article_data["sections"].append(intro_section)
            current_section = {"heading": heading, "paragraphs": []}
            article_data["sections"].append(current_section)
            continue

        # Paragraph-like content
        if el.name in ["p", "ul", "ol", "dd"]:
            text = clean_paragraph(el)
            if text:
                current_section["paragraphs"].append(text)

    # --- Ensure intro is included if it has paragraphs ---
    if intro_section["paragraphs"] and intro_section not in article_data["sections"]:
        article_data["sections"].insert(0, intro_section)

    # --- Categories ---
    catlinks = soup.select("#mw-normal-catlinks ul li a")
    for cat in catlinks:
        article_data["categories"].append(cat.get_text(strip=True))
        href = cat.get("href")
        if href and href.startswith("/wiki/"):
            article_data["category_urls"].append("https://simple.wikipedia.org" + href)

    # --- Ensure at least one section exists ---
    if not article_data["sections"]:
        raise ValueError(f"No sections found for {url}")

    return article_data


In [8]:
test_scrape_simple_wiki_redirect1 = scrape_simple_wiki(url='https://simple.wikipedia.org/wiki/USA') # redirect to 'https://simple.wikipedia.org/wiki/United_States
test_scrape_simple_wiki_redirect1

{'url': 'https://simple.wikipedia.org/wiki/USA',
 'title': 'United States',
 'sections': [{'heading': 'Introduction',
   'paragraphs': ['The United States of America ( USA ), also known as the United States ( U.S. or US ) or colloquially as America, is a country that is mainly in North America. It is made of 50 states, a federal district ( Washington, D.C., the District of Columbia), and some other territories and insular areas. Forty-eight of the states are connected ( Contiguous United States ), and they are bordered by Canada to the north and Mexico to the south. The state of Alaska is in the northwestern area of the continent near Asia ( Russia ) and is separated from the other 48 states by Canada making it an exclave. Alaska is bordered by Canada to its east. The state of Hawaii is a set of islands in the Pacific located within Polynesia and is about 2,200 miles (3,500 kilometers) from the continent. The capital city is Washington, D.C. and the largest city by population is New Yo

In [9]:
# -------- Get Wiki pages URLs from a Wiki Category URL --------------
# Used for scraping all pages in a category instead of scraping them individually.
# Works for both simple and normal wiki category pages.

def get_category_pages(category_url):

    headers = {
        "User-Agent": "YourBot/1.0 (https://example.com/contact)"
    }
    
    try:
         res = requests.get(category_url, headers=headers, timeout=10)
         # Raise an HTTPError for 4xx/5xx responses (e.g., 404, 500, 429), ensuring failed HTTP responses are treated as errors.
         res.raise_for_status()
		
    except requests.RequestException as e:
        # catches all request-related failures: connection errors, timeouts, invalid URLs, and HTTP errors raised by raise_for_status()
		# i.e. network/environment-level failures, not parsing or scraper-logic errors.
        raise RuntimeError(f"Failed to fetch category URL: {category_url}") from e
		
    
    soup = BeautifulSoup(res.text, "html.parser")

    # Extract category name from URL
    url_parse = urlparse(category_url)
    path = url_parse.path
    category_name = path.split(":")[-1]

    base = url_parse.scheme + "://" + url_parse.netloc # https://simple.wikipedia.org
    pages = []

    for li in soup.select("#mw-pages li a"):
        href = li.get("href")
        title = li.get_text(strip=True)
        if "Template:" in title:
            continue
        if href and href.startswith("/wiki/"):
            pages.append({
                "title": title,
                "url": base + href
            })
            
    if not pages:
            raise ValueError(f"No pages found in category {category_name} at {category_url}")

    return {
        "categories": category_name,
        "category_urls": category_url,
        "pages": pages
    }

In [10]:
# test get_category_pages()
simple_category_test = get_category_pages(category_url='https://simple.wikipedia.org/wiki/Category:Movie_producers_from_New_York_City')

# gets only the 200 first pages in the category (wiki category page structure - category has multiple pages if more than 200 pages in the category)
simple_category_test = get_category_pages(category_url='https://simple.wikipedia.org/wiki/Category:Living_people')
# but works if subsecant pages url is provided:
simple_category_test = get_category_pages(category_url='https://simple.wikipedia.org/w/index.php?title=Category:Living_people&pagefrom=Abrines+Redondo%2C+Alejandro%0AAlejandro+Abrines+Redondo#mw-pages')

# works also for normal wiki categories:
normal_category_test = get_category_pages(category_url='https://en.wikipedia.org/wiki/Category:Database_models')

len(simple_category_test['pages']), simple_category_test['pages']

(200,
 [{'title': 'Alejandro Abrines Redondo',
   'url': 'https://simple.wikipedia.org/wiki/Alejandro_Abrines_Redondo'},
  {'title': 'Anne-Ségolène Abscheidt',
   'url': 'https://simple.wikipedia.org/wiki/Anne-S%C3%A9gol%C3%A8ne_Abscheidt'},
  {'title': 'Mohammad Abshak',
   'url': 'https://simple.wikipedia.org/wiki/Mohammad_Abshak'},
  {'title': 'Mehdi Abtahi',
   'url': 'https://simple.wikipedia.org/wiki/Mehdi_Abtahi'},
  {'title': 'Najmeh Abtin',
   'url': 'https://simple.wikipedia.org/wiki/Najmeh_Abtin'},
  {'title': 'Abu Hafs al-Hashimi al-Qurashi',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Hafs_al-Hashimi_al-Qurashi'},
  {'title': 'Abu Haider',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Haider'},
  {'title': 'Eva Abu Halaweh',
   'url': 'https://simple.wikipedia.org/wiki/Eva_Abu_Halaweh'},
  {'title': 'Abu Hamza al-Masri',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Hamza_al-Masri'},
  {'title': 'Abu Khaled',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Khal

In [11]:

# --------- Check if page is already in DB ----------------------------
# ------------- or should be scraped ----------------------------------


def page_needs_scraping(url: str, db_path: str) -> bool:
    """
    Returns True if the page (simple or technical) is NOT yet stored.
    Infers everything from the URL.
    """

    # Extract title
    path = urlparse(url).path
    if "/wiki/" not in path:
        return False  # Not a valid wiki page

    title = unquote(path.split("/wiki/")[-1])

    # Determine which indicator column to check
    if "simple.wikipedia.org" in url:
        indicator_col = "has_simple"
    elif "wikipedia.org" in url:
        indicator_col = "has_technical"
    else:
        return False  # Not supported domain

    # Open connection (thread-safe pattern)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    try:
        cur.execute(
            f"SELECT {indicator_col} FROM pages WHERE title = ?",
            (title,)
        )
        row = cur.fetchone()
    finally:
        conn.close()

    if row is None:
        # Page not in DB at all → needs scraping
        return True

    # If indicator is 0 → needs scraping
    return row[0] == 0

In [12]:

page_needs_scraping(url='https://simple.wikipedia.org/wiki/Markov_chain', db_path="WikipediaOne.db")
page_needs_scraping(url='https://simple.wikipedia.org/wiki/Stochastic_process', db_path="WikipediaOne.db")

True

In [13]:
# ------------ STORAGE FCT TO DB ---------------

def store_page(article_data: dict, conn, cur):
    """
    Stores a scraped Wikipedia article into DB.
    (Updates both tables 'pages' and 'definitions')
    
    article_data = {
        "url": str,
        "title": str,
        "sections": list,
        ...
    }
    """

    url = article_data["url"]
    title = article_data["title"]
    sections = article_data.get("sections", [])
    content = json.dumps(sections)  # store sections as JSON string

    # Determine kind from URL
    kind = "simple" if "simple.wikipedia.org" in url else "technical"

    source = url
    created_at = datetime.now(timezone.utc).isoformat()

    # Ensure page exists (redondance of 'OR IGNORE' as scrape_wikipedia already handles page uniqueness)
    cur.execute(
        "INSERT OR IGNORE INTO pages (title) VALUES (?);",
        (title,)
    )

    # Get page_id
    cur.execute("SELECT id FROM pages WHERE title = ?", (title,))
    page_id = cur.fetchone()[0]

    # Insert definition or update existing 
	# to correct: (scrape_wikipedia() prevents updates though as checks for existance of title first)
    cur.execute(
        """
        INSERT OR REPLACE INTO definitions
        (page_id, kind, content, source, created_at)
        VALUES (?, ?, ?, ?, ?);
        """,
        (page_id, kind, content, source, created_at)
    )

    # Update indicator & commit
    cur.execute(f"UPDATE pages SET has_{kind} = 1 WHERE id = ?", (page_id,))

    conn.commit()


In [14]:
# ---- FRAMEWORK FUNCTION - MAIN SCRAPING FUNCTION ----

def scrape_wikipedia(
        urls: Union[str, List[str]], 
        db_path: Optional[str] = None, 
        cat_workers: int = 5
    ) -> List[dict]:
    """
    General Wikipedia scraper framework.
    
    Parameters
    ----------
    urls : str or List[str]
        Wikipedia page(s) or category URL(s) to scrape.
    db_path : str, optional
        Path to SQLite DB to store results. If None, results are not stored.
    cat_workers : int
        Number of parallel threads for scraping.

    Returns
    -------
    List[dict]
        List of successfully scraped page results.
    
    Notes:
    - Accepts single URL or list of URLs (pages or categories)
    - Automatically detects:
        - Simple vs normal Wikipedia
        - Category vs single page
    - Expands categories to individual page URLs
    - Scrapes pages in parallel using ThreadPoolExecutor
    - Directs to the appropriate scraper function
    """

    if isinstance(urls, str):
        urls = [urls]

    # Expand all category URLs
    expanded_urls = []
    for url in urls:
        if "/wiki/Category:" in url:
            categories = get_category_pages(url)
            pages_urls = [page.get('url', None) for page in categories['pages']]
            expanded_urls.extend(pages_urls)
        else:
            expanded_urls.append(url)
    
    # Decide which need scraping (if db_path provided, don't scrape if already in db)     
    urls_to_scrape = []
    
    for url in expanded_urls:
        if db_path is None:
            urls_to_scrape.append(url)
        else:
            if page_needs_scraping(url, db_path=db_path):
                urls_to_scrape.append(url)
     
    if not urls_to_scrape:
        print("All pages already in DB. Nothing to scrape.")
        return []
    
    
	# Determine which scraper to use for each URL
    def scrape_dispatcher(url: str):
        conn = None
        cur = None
        if db_path:
            conn = sqlite3.connect(db_path)
            cur = conn.cursor()

        try:
            if "simple.wikipedia.org" in url:
                result = scrape_simple_wiki(url)
            else:
                result = scrape_normal_wiki(url)

            if db_path and result:
                store_page(result, conn, cur)

            return result

        except RuntimeError:
            print(f"[WARNING] Skipping URL due to fetch error: {url}")
            return None

        finally: 
            if conn: conn.close()


    # Parallel scraping
    results = []
    with ThreadPoolExecutor(max_workers=cat_workers) as executor:
        futures = [executor.submit(scrape_dispatcher, u) for u in urls_to_scrape]
        for f in futures:
            res = f.result()
            results.append(res)

    return results

In [15]:
scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Data')

[{'url': 'https://simple.wikipedia.org/wiki/Data',
  'title': 'Data',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ['Data is a collection of facts, figures, objects, symbols and events gathered from different sources. Organizations collect data to make better decisions. Without data, it would be difficult for organizations to make appropriate decisions, so data is very important.',
     'Data can exist in different forms:',
     '- Numerical data - Numbers, statistics, measurements\n- Text data - Words, descriptions, documents\n- Visual data - Images, charts, graphs\n- Audio data - Sounds, music, voice recordings\n- Digital data - Information stored in computers',
     'Data helps people understand things better. In computer science, data is information that can be processed by computers. Data is often organized in databases to make it easier to find and use.']},
   {'heading': 'Types of Data',
    'paragraphs': ['There are two main types of data:',
     'Qualitative da

In [16]:
scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Markov_chain', db_path='WikipediaOne.db')

[{'url': 'https://simple.wikipedia.org/wiki/Markov_chain',
  'title': 'Markov chain',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ['A Markov chain is a model of some random process that happens over time. Markov chains are called that because they follow a rule called the Markov property. The Markov property says that whatever happens next in a process only depends on how it is right now (the state). It doesn\'t have a "memory" of how it was before. It is helpful to think of a Markov chain as evolving through discrete steps in time, although the "step" doesn\'t need to have anything to do with time.',
     "Markov chains can be discrete or continuous. Discrete Time Markov Chains are split up into discrete time steps, like t = 1, t = 2, t = 3, and so on. The probability that a chain will go from one state to another state depends only on the state that it's in right now. Continuous Time Markov Chains are chains where the time spent in each state is a real number. The am

In [17]:
scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Stochastic_process', db_path='WikipediaOne.db')

[{'url': 'https://simple.wikipedia.org/wiki/Stochastic_process',
  'title': 'Stochastic process',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ['A Stochastic process is a mathematical description of random events that occur one after another. It is possible to order these events according to the time at which they occur.',
     "This can be used to model such things as daily weather data, or exchange rate changes, or medical information like a patient's EKG, EEG, blood pressure or temperature.",
     'Stochastic processes used in various disciplines, including physics, biology, finance, telecommunications, and operations research. They provide a powerful framework for analyzing and predicting the behavior of systems under uncertain conditions.']}],
  'categories': ['Statistics'],
  'category_urls': ['https://simple.wikipedia.org/wiki/Category:Statistics']}]

In [18]:
test1 = ['https://simple.wikipedia.org/wiki/Stochastic_process',
         'https://simple.wikipedia.org/wiki/Category:Statistics',]

test1_res = scrape_wikipedia(urls=test1, db_path='WikipediaOne.db')
len(test1_res), test1_res

(82,
 [{'url': 'https://simple.wikipedia.org/wiki/Stochastic_process',
   'title': 'Stochastic process',
   'sections': [{'heading': 'Introduction',
     'paragraphs': ['A Stochastic process is a mathematical description of random events that occur one after another. It is possible to order these events according to the time at which they occur.',
      "This can be used to model such things as daily weather data, or exchange rate changes, or medical information like a patient's EKG, EEG, blood pressure or temperature.",
      'Stochastic processes used in various disciplines, including physics, biology, finance, telecommunications, and operations research. They provide a powerful framework for analyzing and predicting the behavior of systems under uncertain conditions.']}],
   'categories': ['Statistics'],
   'category_urls': ['https://simple.wikipedia.org/wiki/Category:Statistics']},
  {'url': 'https://simple.wikipedia.org/wiki/Regression_toward_the_mean',
   'title': 'Regression tow

In [19]:
test2 = ['https://en.wikipedia.org/wiki/Category:Database_models',
         'https://en.wikipedia.org/wiki/Heterogeneous_database_system',]

test2_res = scrape_wikipedia(urls=test2, db_path='WikipediaOne.db')
len(test2_res), test2_res

(13,
 [{'url': 'https://en.wikipedia.org/wiki/Database_model',
   'title': 'Database model',
   'sections': [{'heading': 'Introduction',
     'paragraphs': ['A database model is a type of data model that determines the logical structure of a database. It fundamentally determines in which manner data can be stored, organized and manipulated. The most popular example of a database model is the relational model, which uses a table-based format.']},
    {'heading': 'Types',
     'paragraphs': ['Common logical data models for databases include:',
      '- Hierarchical database model',
      'Hierarchical database model',
      'This is the oldest form of database model. It was developed by IBM for IMS (information Management System), and is a set of organized data in tree structure. DB record is a tree consisting of many groups called segments. It uses one-to-many relationships, and the data access is also predictable.',
      '- Network model\n- Relational model\n- Entity–relationship mode

In [20]:
scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Blockchain', db_path='WikipediaOne.db')

All pages already in DB. Nothing to scrape.


[]

In [21]:

# ------------------------ BACKFILL DB ------------------------
# ----- with existing simple/technical page counterparts ------


def backfill_DB(
        db_path: str,
        cat_workers: int = 5
    ) -> List[dict]:
    """
    Checks DB for pages where:
        - has_simple = 0
        - has_technical = 0
    
    Attempts to construct the missing URL version (works only if direct mapping between urls) and scrape it.
    
    Returns list of successfully scraped pages.
    """

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # Fetch all pages
    cur.execute("""
        SELECT id, title, has_simple, has_technical
        FROM pages
        WHERE has_simple = 0 OR has_technical = 0;

    """)
    rows = cur.fetchall()
    conn.close()
    
    urls_to_scrape = []
    
    for _, title, has_simple, has_technical in rows:
        if has_simple == 0:
            # construct canonical technical or simple URLs and add to scraping list
            simple_url = f"https://simple.wikipedia.org/wiki/{title.replace(' ', '_')}"
            urls_to_scrape.append(simple_url)
        if has_technical == 0:
            normal_url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
            urls_to_scrape.append(normal_url)
      

    if not urls_to_scrape:
        print("No missing wiki versions found.")
        return []

    print(f"Attempting to scrape {len(urls_to_scrape)} missing versions...")

    results = scrape_wikipedia(
        urls=urls_to_scrape,
        db_path=db_path,
        cat_workers=cat_workers
    )

    return results # some elem in list results might be None but scrape_wikipedia() does not store them


In [22]:
test_backfill_DB1 = backfill_DB(db_path='WikipediaOne.db')
len(test_backfill_DB1), test_backfill_DB1

Attempting to scrape 33 missing versions...
[WARNING] Skipping URL due to fetch error: https://en.wikipedia.org/wiki/Inference_(statistics)
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Array_DBMS
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Data_integration
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Database_model
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Component-oriented_database
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Data_orientation
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Entity–attribute–value_model
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Object_database
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Heterogeneous_database_system
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Netw

(33,
 [{'url': 'https://en.wikipedia.org/wiki/Bose-Einstein_statistics',
   'title': 'Bose–Einstein statistics',
   'sections': [{'heading': 'Introduction',
     'paragraphs': ['- Thermodynamics\n- Kinetic theory',
      'Thermodynamics',
      'Kinetic theory',
      'In quantum statistics, Bose–Einstein statistics ( B–E statistics ) describes one of two possible ways in which a collection of non-interacting identical particles may occupy a set of available discrete energy states at thermodynamic equilibrium. The aggregation of particles in the same state, which is a characteristic of particles obeying Bose–Einstein statistics, accounts for the cohesive streaming of laser light and the frictionless creeping of superfluid helium. The theory of this behaviour was developed (1924–25) by Satyendra Nath Bose, who recognized that a collection of identical and indistinguishable particles could be distributed in this way. The idea was later adopted and extended by Albert Einstein in collabora

In [23]:
[elem.get('title','') for elem in test_backfill_DB1 if elem != None]

['Bose–Einstein statistics',
 'Demography',
 'Frequency (statistics)',
 'Frequentist probability',
 'Dependent and independent variables',
 'Independence (probability theory)',
 'Maximum likelihood estimation',
 'Statistical population',
 'Sampling (statistics)',
 'Sample size determination',
 'Survey methodology',
 'Time series',
 'Bose-Einstein statistics',
 'Population (statistics)',
 'Statistical survey']

In [ ]:
# ----------------- Definition for Kids - Generation ---------------

from openai import OpenAI
from datetime import datetime

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)

MAX_INPUT_TOKENS = 16384 # model's (Llama-3.1-8B-Instruct) context length (16384 tokens).

data_path = os.getcwd()
db_name = 'WikipediaOne.db'
db_path = data_path + '/' + db_name



def generate_and_store_kids_definition_tuple(page_tuple, db_path, client, max_input_tokens=16384):
    """
    page_tuple: (page_id, title, content)
    """
    page_id, title, description = page_tuple

    description = description[:max_input_tokens]

    # Generate kids definition
    try:
        completion = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct:novita",
            messages=[
                {"role": "system", "content": (
                    "Explain concepts clearly for children around 10 years old. "
                    "Use simple words, short sentences, and concrete examples. "
                    "Avoid technical terms unless they are explained. "
                    "Do not include introductions, titles, or meta commentary. "
                    "Output only the explanation."
                )},
                {"role": "user", "content": f"Explain this so a 10-year-old can understand it:\n\ntopic: {title}\ndescription: {description}"}
            ],
            max_completion_tokens=500,
        )
        kids_definition = completion.choices[0].message.content
    except Exception as e:
        print(f"[WARNING] Skipping {title} due to API error: {e}")
        kids_definition = None

    # Store in DB
    if kids_definition:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        try:
            cur.execute("""
                INSERT OR REPLACE INTO definitions
                (page_id, kind, content, source, created_at)
                VALUES (?, 'kids', ?, 'generated_by_LLM', ?)
            """, (page_id, kids_definition, datetime.now(timezone.utc).isoformat()))

            cur.execute("UPDATE pages SET has_kids = 1 WHERE id=?", (page_id,))
            conn.commit()
        finally:
            conn.close()

    return kids_definition



In [34]:
# ----------------- Definition for Kids - Generation ---------------

from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)

MAX_INPUT_TOKENS = 16384 # model's (Llama-3.1-8B-Instruct) context length (16384 tokens).

data_path = os.getcwd()
db_name = 'WikipediaOne.db'
db_path = data_path + '/' + db_name

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Fetch fetch pages content to feed the llm 
# fetching logic: use first simple definitions (lower nb token + less complex text for a simple model)
cur.execute("""
	SELECT
		p.id,
		p.title,
		CASE
			WHEN p.has_simple = 1 THEN s.content
			ELSE t.content
		END AS selected_content
	FROM pages p
	LEFT JOIN definitions s ON p.id = s.page_id AND s.kind = 'simple'
	LEFT JOIN definitions t ON p.id = t.page_id AND t.kind = 'technical'
	WHERE p.has_simple = 1 OR p.has_technical = 1;
""")

data4gen = cur.fetchall()
conn.close()

data4gen[4:6]





[(5,
  'Regression toward the mean',
  '[{"heading": "Introduction", "paragraphs": ["Regression toward the mean simply means that, following an extreme random event, the next random event is likely to be less extreme. Regression toward the mean was first described by Francis Galton. He found that offspring of tall parents tended to be shorter. Also, offspring of shorter parents tended to be taller. Galton stated that processes that did not follow regression towards the mean would quickly go out of control."]}, {"heading": "History", "paragraphs": ["In 1886, Galton published a paper called Regression towards mediocrity in hereditary stature. In the paper, he observed that extreme characteristics (e.g., height) in parents are not passed on completely to their offspring. Rather, the characteristics in the offspring regress towards a mediocre point. Today, this point is called the mean. By measuring the heights of hundreds of people, he was able to quantify regression to the mean, and esti

In [ ]:
# ----------------- Definition for Kids - Generation ---------------

from openai import OpenAI
from datetime import datetime

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)

MAX_INPUT_TOKENS = 16384 # model's (Llama-3.1-8B-Instruct) context length (16384 tokens).

data_path = os.getcwd()
db_name = 'WikipediaOne.db'
db_path = data_path + '/' + db_name


# generation of definitions for kids using a LLM, based on scraped wikipedia page content (use simple definition if available)
def generate_kids_definition(page_tuple, client, max_input_tokens=16384):
    """
    page_tuple: (page_id, title, content)
    """
    page_id, title, description = page_tuple

    description = description[:max_input_tokens]

    # Generate kids definition
    try:
        completion = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct:novita",
            messages=[
                {"role": "system", "content": (
                    "Explain concepts clearly for children around 10 years old. "
                    "Use simple words, short sentences, and concrete examples. "
                    "Avoid technical terms unless they are explained. "
                    "Do not include introductions, titles, or meta commentary. "
                    "Output only the explanation."
                )},
                {"role": "user", "content": f"Explain this so a 10-year-old can understand it:\n\ntopic: {title}\ndescription: {description}"}
            ],
            max_completion_tokens=500,
        )
        kids_definition = completion.choices[0].message.content
    except Exception as e:
        print(f"[WARNING] Skipping {title} due to API error: {e}")
        kids_definition = None

    return (page_id, kids_definition)


generate_kids_definition(data4gen[1], client=client)





"Imagine you're playing a game where you roll a dice and then roll it again after a few minutes. Each time you roll the dice, you can get a different number. \n\nA Stochastic process is like a long list of all the numbers you get when you roll the dice many times, in the order that you rolled them. It's a way to describe what happens when we can't be sure what will happen next, but we can see what happened before.\n\nFor example, if you're trying to predict the weather, you can look at what the weather was like yesterday, last week, and last year. That's like a Stochastic process. It helps us understand what might happen next by looking at what happened before."

In [36]:
# ----- fetching data from db for kid definition generation ------

# Config
data_path = os.getcwd()
db_name = 'WikipediaOne.db'
db_path = data_path + '/' + db_name

# Fetching function to fetch all pages content to feed the llm 
def fetch4kidgen(db_path: str):
	conn = sqlite3.connect(db_path)
	cur = conn.cursor()

	# fetching logic: use first simple definitions (lower nb token + less complex text for a simple model)
	cur.execute("""
		SELECT
			p.id,
			p.title,
			CASE
				WHEN p.has_simple = 1 THEN s.content
				ELSE t.content
			END AS selected_content
		FROM pages p
		LEFT JOIN definitions s ON p.id = s.page_id AND s.kind = 'simple'
		LEFT JOIN definitions t ON p.id = t.page_id AND t.kind = 'technical'
		WHERE p.has_simple = 1 OR p.has_technical = 1;
	""")

	data4gen = cur.fetchall()
	conn.close()

	return data4gen

data4gen = fetch4kidgen(db_path=db_path)
data4gen[4:6]

[(5,
  'Regression toward the mean',
  '[{"heading": "Introduction", "paragraphs": ["Regression toward the mean simply means that, following an extreme random event, the next random event is likely to be less extreme. Regression toward the mean was first described by Francis Galton. He found that offspring of tall parents tended to be shorter. Also, offspring of shorter parents tended to be taller. Galton stated that processes that did not follow regression towards the mean would quickly go out of control."]}, {"heading": "History", "paragraphs": ["In 1886, Galton published a paper called Regression towards mediocrity in hereditary stature. In the paper, he observed that extreme characteristics (e.g., height) in parents are not passed on completely to their offspring. Rather, the characteristics in the offspring regress towards a mediocre point. Today, this point is called the mean. By measuring the heights of hundreds of people, he was able to quantify regression to the mean, and esti

In [ ]:
# Generate all definitions in parallel
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(generate_kids_definition, data4gen))

In [ ]:
# STORING FUNCTION

def store_kids(kid_defs: list, db_path: str):
	"""
    kid_defs: List
			List of tuples. tuples: (page_id: int, kids_definition: str)
    """
	conn = sqlite3.connect(db_path)
	cur = conn.cursor()
	for page_id, kid_def in kid_defs:
		if kid_def:
			cur.execute("""
				INSERT OR REPLACE INTO definitions
				(page_id, kind, content, source, created_at)
				VALUES (?, 'kids', ?, 'generated_by_LLM', ?)
			""", (page_id, kid_def, datetime.now(timezone.utc).isoformat()))
			cur.execute("UPDATE pages SET has_kids = 1 WHERE id=?", (page_id,))
	conn.commit()
	conn.close()

In [ ]:
# FULL FUNCTION